# 02 — Feature Extraction

**Goal**: Extract five types of NLP features from earnings call transcripts.

| Method | Session | Type |
|--------|---------|------|
| Loughran–McDonald Lexicon | 11 | Rule-based sentiment |
| N-grams | 3 | Bag-of-words frequency |
| TF-IDF | 7 | Weighted bag-of-words |
| Word2Vec | 13 | Dense word embeddings |
| FinBERT | 19-20 | Transformer sentiment |

In [ ]:
import os, sys
from pathlib import Path

# Navigate to project root so relative paths work correctly
# When running from notebooks/, we need to go up one level
project_root = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd()
# Fallback: search upward for pyproject.toml
for p in [project_root] + list(project_root.parents):
    if (p / "pyproject.toml").exists():
        project_root = p
        break
os.chdir(project_root)
# Add src/ to Python path so nasdaq_nlp imports work even without install
sys.path.insert(0, str(project_root / "src"))
print(f"Working directory: {Path.cwd()}")

## 1. Preprocessing

Before extracting features, we clean each transcript:
1. **Strip header**: remove participant lists, operator instructions, boilerplate
2. **Section split**: separate Presentation (exec remarks) from Q&A (analyst dialog)
3. **Normalise**: lowercase, remove punctuation

We do NOT remove stopwords for lexicon/FinBERT (need complete words).
For TF-IDF and Word2Vec, we remove stopwords to focus on content words.

In [ ]:
from nasdaq_nlp.data.loader import scan_transcripts
from nasdaq_nlp.preprocessing.text import preprocess_transcript

records = scan_transcripts()
# Load and preprocess the first transcript as a demo
records[0].load()
result = preprocess_transcript(records[0].raw_text)

print("=== Presentation section (first 300 chars) ===")
print(result['presentation_raw'][:300])
print()
print("=== Q&A section (first 300 chars) ===")
print(result['qa_raw'][:300])
print()
print(f"Total tokens (full transcript): {len(result['tokens'])}")

## 2. Loughran–McDonald Lexicon Sentiment

**Math:**

$$\text{NegRate}_i = \frac{\#\text{negative words in transcript } i}{\text{total words}}$$
$$\text{PosRate}_i = \frac{\#\text{positive words in transcript } i}{\text{total words}}$$

These rates are our primary sentiment features. We normalise by total words
to control for transcript length (a 12,000-word call would naturally have
more negative words than an 8,000-word call, even at the same *rate*).

In [ ]:
from nasdaq_nlp.features.lexicon import build_lexicon_features

lexicon_df = build_lexicon_features()
print(lexicon_df[['ticker','neg_rate','pos_rate','total_tokens']].describe().round(4))
print()
print("Top 5 most negative calls:")
print(lexicon_df.nlargest(5, 'neg_rate')[['ticker','neg_rate','pos_rate']].to_string())

## 3. N-gram Features  (Session 3)

An n-gram is a contiguous sequence of n words.

- **Unigrams** (n=1): `['revenue', 'growth', 'exceeded']`
- **Bigrams** (n=2): `['revenue growth', 'growth exceeded']`

We count how often each n-gram appears in each transcript (bag-of-words).
The vocabulary is restricted to the top 500 n-grams by frequency.

Bigrams capture negation (`not strong`) and collocations (`market share`).

In [ ]:
from pathlib import Path
from nasdaq_nlp.data.loader import scan_transcripts
from nasdaq_nlp.features.ngrams import build_ngram_matrix, get_top_ngrams

records = scan_transcripts()
file_paths = [r.file_path for r in records]

X_ng, features_ng, vec_ng = build_ngram_matrix(file_paths, ngram_range=(1,2), max_features=500)
print(f"N-gram matrix shape: {X_ng.shape}  (documents × features)")
print()
print("Top 20 n-grams by corpus frequency:")
print(get_top_ngrams(vec_ng, X_ng, top_n=20).to_string(index=False))

## 4. TF-IDF  (Session 7)

TF-IDF weights each word by how important it is to a specific document,
relative to the whole corpus.

$$\text{TF-IDF}(t, d) = \underbrace{\frac{\text{count}(t, d)}{|d|}}_{\text{TF}} \times \underbrace{\log\frac{N}{df(t)}}_{\text{IDF}}$$

- **TF**: how often term $t$ appears in document $d$
- **IDF**: inverse document frequency — high if $t$ is rare across all $N$ documents

A high TF-IDF score means the term is frequent in *this* document but rare elsewhere
→ it characterises this document specifically.

In [ ]:
from nasdaq_nlp.features.tfidf import build_tfidf_features, top_tfidf_terms_by_ticker
import pandas as pd

tfidf_df, vectorizer = build_tfidf_features()
print(f"TF-IDF feature matrix: {tfidf_df.shape}")
print()

events = pd.read_csv('outputs/processed/event_study_dataset.csv')
X_tfidf = tfidf_df.filter(like='tfidf_').values
top_terms = top_tfidf_terms_by_ticker(events, vectorizer, X_tfidf, top_n=5)
print("Top 5 characteristic terms by ticker:")
print(top_terms.to_string(index=False))

## 5. Word2Vec Document Embeddings  (Session 13)

Word2Vec learns a dense vector representation for each word by training a
neural network to predict surrounding words (skip-gram architecture).

**Key property**: semantically similar words cluster together:
$$\text{cosine}(\vec{\text{strong}}, \vec{\text{robust}}) \approx 1$$

To represent an entire transcript (document), we **average-pool** word vectors:
$$\vec{d} = \frac{1}{|tokens|} \sum_{w \in d} \vec{w}$$

Each document is now a 100-dimensional dense vector, capturing overall semantic content.

In [ ]:
from nasdaq_nlp.features.embeddings import build_embedding_features, nearest_neighbors

emb_df, w2v_model = build_embedding_features()
print(f"Embedding matrix: {emb_df.shape}")
print()
# Sanity check: nearest neighbors for financial terms
for word in ['growth', 'risk', 'revenue', 'strong', 'guidance']:
    neighbors = nearest_neighbors(w2v_model, word, top_n=5)
    if neighbors:
        nn_str = ', '.join(f"{w}({s:.2f})" for w, s in neighbors)
        print(f"  '{word}' → {nn_str}")

## 6. FinBERT Sentiment  (Sessions 19-20)

FinBERT is BERT fine-tuned on financial text. Unlike the lexicon, it understands
context (negation, idioms, domain jargon).

For each sentence in the transcript:
$$P(\text{positive}), P(\text{negative}), P(\text{neutral}) \quad \text{(sum to 1)}$$

We average across all sentences in the transcript:
$$\bar{P}(\text{negative}) = \frac{1}{S}\sum_{s=1}^{S} P_s(\text{negative})$$

In [ ]:
from pathlib import Path
finbert_path = Path('outputs/processed/finbert_features.csv')

if finbert_path.exists():
    import pandas as pd
    fb = pd.read_csv(finbert_path)
    print(f"FinBERT features: {len(fb)} events")
    print(fb[['ticker','finbert_pos_mean','finbert_neg_mean','finbert_neu_mean']].describe().round(3))
else:
    print("FinBERT features not yet computed.")
    print("Run: from nasdaq_nlp.features.finbert import build_finbert_features; build_finbert_features()")
    print("(Takes ~20-30 min on CPU)")

In [ ]:
# Verification: feature extraction complete
import numpy as np
assert tfidf_df.shape == (188, 503), f"Unexpected TF-IDF shape: {tfidf_df.shape}"
assert emb_df.shape == (188, 103), f"Unexpected embedding shape: {emb_df.shape}"
print("✓ All feature extraction steps verified")
print(f"  Lexicon:    {len(lexicon_df)} events × 5 features")
print(f"  TF-IDF:     {tfidf_df.shape[0]} events × {tfidf_df.shape[1]-3} features")
print(f"  Word2Vec:   {emb_df.shape[0]} events × {emb_df.shape[1]-3} dimensions")